# Running the online simulation of all classifiers

This notebook contains the code to run an online simulation for every classifier proposed in the thesis.

In [ ]:
# imports
import pickle
import mne
import warnings
from utils.run_patient_v3 import run_patient_online_sessions_window_v5, run_patient_online_sessions_CC, run_patient_online_sessions_aphasia_slda, run_patient_online_sessions_static, run_patient_online_sessions_window_v4, run_patient_online_sessions_transfer_v2
from utils.db import patients_db
from utils.static_protocol import static_protocol

# Turn off warnings (that most likely occur from ToeplitzLDA)
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
mne.set_log_level('WARNING')



In [ ]:
# Run online simulation of Adaptive CC BT-LDA on all patients with the optimal UC-pair from even patients
# To obtain the data for the odd patients, replace "even" by "odd"

group = "even" # or "odd"

if group == "even":
# optimal UC-pair obtained from odds, used for evens to avoid overfitting
    UC_mean = 0.5 * (2.0 **-7)
    UC_cov = 0.5 * (2.0 **-13)
else:
# optimal UC-pair for evens, likewise
    UC_mean = 0.5 * (2.0 **-7)
    UC_cov = 0.5 * (2.0 **-11)

print(UC_mean)
print(UC_cov)

for id in patients_db:
    info = patients_db.get(id)
    patient = info.get('patient_nr')
    last_session = info.get('last_session')
    calibration_selection = info.get('selection')

    print("patient: ", patient)
    print("last session", last_session)
    print("calibration_selection", calibration_selection)

    performances = run_patient_online_sessions_CC(patient=patient, last_session_nr=last_session, calibration_selection=calibration_selection, UC_mean=UC_mean, UC_cov=UC_cov, version=2)
    with open(f"p{patient}_cc_uc_{group}.pkl", 'wb') as f: 
        pickle.dump(performances, f)

In [ ]:
# Run Adaptive CC sLDA [1] on all patients
#
# [1] M. Musso et al., “Aphasia recovery by language training using a brain–computer interface: a proof-of-concept study,” Brain Communications, vol. 4, no. 1, p. fcac008, Feb. 2022, doi: 10.1093/braincomms/fcac008.

# selected UC-pair [1]
UC_mean = 0.005
UC_cov = 0.001

for id in static_protocol:
    info = static_protocol.get(id)
    patient = info.get('patient_nr')
    last_session = info.get('last_session')
    calibration_selection = info.get('selection')
    changing_conditions = info.get('changing_condition')

    if changing_conditions:
        starter_conditions = info.get('changing_starter_sessions')
    else:
        starter_conditions = None
        
    print("patient: ", patient)
    print("last session: ", last_session)
    print("calibration_selection: ", calibration_selection)
    print("changing conditions: ", changing_conditions)

    performances = run_patient_online_sessions_aphasia_slda(patient, last_session, calibration_selection, starter_conditions, UC_mean=UC_mean, UC_cov=UC_cov)
    
    with open(f"p{patient}_aphasia_slda.pkl", 'wb') as f: 
        pickle.dump(performances, f)


In [ ]:
# Run Transfer Fixed BT-LDA on all patients

for id in patients_db:
    info = patients_db.get(id)
    patient = info.get('patient_nr')
    last_session = info.get('last_session')
    calibration_selection = info.get('selection')

    print("patient: ", patient)
    print("last session", last_session)
    print("calibration_selection", calibration_selection)

    performances = run_patient_online_sessions_transfer_v2(patient, last_session, calibration_selection)
    with open(f"p{patient}_transfer_v2.pkl", 'wb') as f: 
        pickle.dump(performances, f)

In [ ]:
# Run Static Fixed BT-LDA on all patients

for id in static_protocol:
    info = static_protocol.get(id)
    patient = info.get('patient_nr')
    last_session = info.get('last_session')
    calibration_selection = info.get('selection')
    changing_conditions = info.get('changing_condition')
    if changing_conditions:
        starter_conditions = info.get('changing_starter_sessions')
    else:
        starter_conditions = None
        
    print("patient: ", patient)
    print("last session: ", last_session)
    print("calibration_selection: ", calibration_selection)
    print("changing conditions: ", changing_conditions)

    performances = run_patient_online_sessions_static(patient, last_session, calibration_selection, starter_conditions)
    with open(f"p{patient}_performances_static_v3.pkl", 'wb') as f: 
        pickle.dump(performances, f)

In [ ]:
# Run Adaptive Window BT-LDA on all patients
# v5: fixed window size = min(3600, previous_session_size) - time ivals 0.1-0.81, 50ms

for id in patients_db:
    info = patients_db.get(id)
    patient = info.get('patient_nr')
    last_session = info.get('last_session')
    calibration_selection = info.get('selection')

    print("patient: ", patient)
    print("last session", last_session)
    print("calibration_selection", calibration_selection)

    performances = run_patient_online_sessions_window_v5(patient, last_session, calibration_selection)
    with open(f"p{patient}_window_v5.pkl", 'wb') as f: 
        pickle.dump(performances, f)

In [ ]:
# Run modified Adaptive Window v4 on all
# v4: window size = previous session - time ivals 0.1-0.81, 50ms
# this version was not used in the thesis

for id in patients_db:
    info = patients_db.get(id)
    patient = info.get('patient_nr')
    last_session = info.get('last_session')
    calibration_selection = info.get('selection')

    print("patient: ", patient)
    print("last session", last_session)
    print("calibration_selection", calibration_selection)

    performances = run_patient_online_sessions_window_v4(patient, last_session, calibration_selection)
    with open(f"p{patient}_window_v3.pkl", 'wb') as f: 
        pickle.dump(performances, f)